In [1]:
import gurobipy as gp
from gurobipy import GRB
import itertools
from collections.abc import Mapping, Sequence
import numpy as np

from instance import Instance, Arc, Node

instance = Instance()
world = instance.world
scenarios = instance.scenarios
demands = instance.demands
paths_per_od = instance.paths_per_od
active_arcs = instance.active_arcs
arc_to_idx = instance.arc_to_idx
tau = instance.tau
phi = instance.phi

model = gp.Model("asymmetric dictator (anonymous)")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

Set parameter Username
Set parameter LicenseID to value 2844113
Academic license - for non-commercial use only - expires 2027-07-13


In [2]:
# decision: x_od[omega, od, a] := total flow on arc a under scenario omega for a given OD pair
x_od = {
    (scenario_name, od, a): model.addVar(vtype=GRB.INTEGER, lb=0, ub=demand, name=f"x_od_{scenario_name}_{od}_{a}")
    for od, demand in demands.items()
    for a in active_arcs
    for scenario_name in scenarios
}

# decision: x[omega, a] := total flow on arc a under scenario omega
x = {
    (scenario_name, a): model.addVar(vtype=GRB.INTEGER, lb=0, ub=n, name=f"x_{scenario_name}_{a}")
    for a in active_arcs
    for scenario_name in scenarios
}

# constraint: flow conservation
for scenario_name in scenarios:
    for od, demand in demands.items():
        for v in V:
            flow_out = gp.quicksum(x_od[scenario_name, od, a] for a in active_arcs if a[0] == v)
            flow_in = gp.quicksum(x_od[scenario_name, od, a] for a in active_arcs if a[1] == v)
            flow = demand if v == od[0] else (-demand if v == od[1] else 0)

            model.addConstr(flow_out - flow_in == flow, name=f"flow_{scenario_name}_{od}_{v}")

# constraint: x[omega, a] = \sum_{t \in T} x_od[omega, t, a]
for scenario_name in scenarios:
    for a in active_arcs:
        model.addConstr(
            x[scenario_name, a] == gp.quicksum(x_od[scenario_name, od, a] for od in demands),
            name=f"x_{a}"
        )

k_vals = np.arange(n + 1)
# objective: \min \sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} x[\omega, a] \cdot tau[\omega, a, x[a]]
for scenario_name, mu in scenarios.items():
    for i, arc in enumerate(active_arcs):
        costs = mu * k_vals * tau[scenario_name][i, :n + 1]
        model.setPWLObj(
            x[scenario_name, arc],
            k_vals.tolist(),
            costs.tolist()
        )

In [3]:
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

In [4]:
def debug():
    print(f"{model.ObjVal=:.2f}")
    print(f"{model.Runtime=:.3f}")

    for scenario_name in scenarios:
        print(f"\nScenario: {scenario_name}")
        for od, demand in demands.items():
            print(f"\t{od}:")

            flows = {a: flow for a in active_arcs if (flow:=x_od[scenario_name, od, a].X) > 0}

            # greedily decompose into paths
            for _ in range(demand):
                path = [od[0]]
                curr = od[0]

                while curr != od[1]: # NOTE: assuming flow conservation from above
                    for a, flow in flows.items():
                        if a[0] == curr and flow > 0:
                            if flow > 1:
                                flows[a] -= 1
                            else:
                                del flows[a]
                            curr = a[1]
                            path.append(curr)
                            break

                print(f"\t\t{path}")

print("OPTIMAL")
optimal_social_cost = model.ObjVal
print(f"{optimal_social_cost=:.2f}")
debug()

OPTIMAL
optimal_social_cost=174.04
model.ObjVal=174.04
model.Runtime=0.027

Scenario: nominal
	((1, 0), (1, 3)):
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3)]
	((3, 0), (3, 3)):
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (3, 3)]
		[(3, 0), 

In [5]:
# objective: TODO
for scenario_name, mu in scenarios.items():
    for i, arc in enumerate(active_arcs):
        potentials = mu * phi[scenario_name][i, :n + 1]
        model.setPWLObj(
            x[scenario_name, arc],
            k_vals.tolist(),
            potentials.tolist()
        )

model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

selfish_social_cost = 0
for scenario_name, mu in scenarios.items():
    for i, arc in enumerate(active_arcs):
        k = int(round(x[scenario_name, arc].X))
        selfish_social_cost += mu * k * tau[scenario_name][i, k]

print("SELFISH")
PoA = selfish_social_cost / optimal_social_cost
print(f"{selfish_social_cost=:.2f}")
print(f"{PoA=:.5f}")
debug()

SELFISH
selfish_social_cost=193.52
PoA=1.11193
model.ObjVal=156.96
model.Runtime=0.003

Scenario: nominal
	((1, 0), (1, 3)):
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
	((3, 0), (3, 3)):
		[(3, 0), (3, 1), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (3, 1), (3, 2), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3)]
		[(3, 0), (2, 